[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gdslab/d2spy/blob/main/docs/guides/notebooks/10_working_with_annotations.ipynb)
[![Jupyter Notebook](https://img.shields.io/badge/Open%20in%20JuypterHub%20-%20%233776AB?logo=jupyter&logoColor=%23F37626&labelColor=%23F5F5F5)](https://lab.d2s.org/hub/user-redirect/lab/tree/tutorials/10_working_with_annotations.ipynb)

# Working with annotations on data products
*This guide will walk you through creating, retrieving, filtering, updating, and deleting annotations on your Data to Science (D2S) data products. You will also learn how to upload and manage annotation attachments.*

To get started, you will need to import the `Workspace` module.

In [ ]:
# Uncomment and run the following line if working out of Google Colab
# !pip install d2spy

In [ ]:
from d2spy.workspace import Workspace

All of your D2S data can be accessed through a D2S "workspace." The [`Workspace`](https://py.d2s.org/workspace/) module's [`connect`](https://py.d2s.org/workspace/#d2spy.workspace.Workspace.connect) method can be used to login to a D2S instance and connect to your workspace in one go. Behind the scenes, the [`Auth`](https://py.d2s.org/auth/) module will be used to handle authenticating with D2S and requesting an authorization token. You will need to provide `connect` with the URL to your D2S instance and enter your password when prompted.

In [ ]:
# Example of connecting to a workspace for a local D2S instance
workspace = Workspace.connect("http://localhost:8000", "yourD2Semail@example.com")

## Navigating to a data product

Annotations are associated with data products. Before working with annotations, you need to navigate to the data product you want to annotate. See the second guide, [Accessing your workspace projects, flights, and data products](https://py.d2s.org/guides/notebooks/02_accessing_your_workspace/), for detailed examples using these methods.

In [ ]:
# Get your projects
projects = workspace.get_projects()

# Get flights for the first project
flights = projects[0].get_flights()

# Get data products for the first flight
data_products = flights[0].get_data_products()

# Select the data product you want to annotate
data_product = data_products[0]
print(data_product)

## Adding an annotation

The DataProduct [`add_annotation`](https://py.d2s.org/data_product/#d2spy.models.data_product.DataProduct.add_annotation) method creates a new annotation on the data product. You must provide a `description` and a `geom` (a GeoJSON Feature). You can optionally include `tags` (a list of tag names), `visibility` (`"owner"` or `"project"`), and `style` (a dict of custom styling metadata).

In [ ]:
# Create a GeoJSON Feature for the annotation geometry
annotation_geom = {
    "type": "Feature",
    "geometry": {
        "type": "Polygon",
        "coordinates": [
            [
                [-86.944981783977838, 41.444435853085622],
                [-86.943319754949272, 41.444435046238446],
                [-86.94332056379109, 41.443505552529658],
                [-86.944982569102066, 41.443506359350643],
                [-86.944981783977838, 41.444435853085622],
            ]
        ],
    },
    "properties": {},
}

# Add annotation with tags and default visibility ("owner")
annotation = data_product.add_annotation(
    description="Field boundary annotation",
    geom=annotation_geom,
    tags=["boundary", "field-1"],
)
print(annotation)

You can set `visibility` to `"project"` to make the annotation visible to all project members.

In [ ]:
# Add annotation visible to all project members
shared_annotation = data_product.add_annotation(
    description="Damage area noted during field visit",
    geom=annotation_geom,
    tags=["damage"],
    visibility="project",
)
print(shared_annotation)

## Retrieving annotations

The DataProduct [`get_annotations`](https://py.d2s.org/data_product/#d2spy.models.data_product.DataProduct.get_annotations) method retrieves all annotations for a data product as an [`AnnotationCollection`](https://py.d2s.org/annotation_collection/). You can also retrieve a single annotation by its ID using [`get_annotation`](https://py.d2s.org/data_product/#d2spy.models.data_product.DataProduct.get_annotation).

In [ ]:
# Get all annotations for the data product
annotations = data_product.get_annotations()
print(f"Total annotations: {len(annotations)}")

# Print each annotation
for ann in annotations:
    print(ann)

In [ ]:
# Get a single annotation by its ID
annotation_id = str(annotations[0].id)
single_annotation = data_product.get_annotation(annotation_id)
print(single_annotation)

## Filtering annotations

The `annotations` variable is an [`AnnotationCollection`](https://py.d2s.org/annotation_collection/). The collection can be filtered by tag using [`filter_by_tag`](https://py.d2s.org/annotation_collection/#d2spy.models.annotation_collection.AnnotationCollection.filter_by_tag) or by visibility using [`filter_by_visibility`](https://py.d2s.org/annotation_collection/#d2spy.models.annotation_collection.AnnotationCollection.filter_by_visibility).

In [ ]:
# Filter annotations by tag
boundary_annotations = annotations.filter_by_tag("boundary")
print(f"Annotations tagged 'boundary': {len(boundary_annotations)}")

In [ ]:
# Filter annotations by visibility
project_annotations = annotations.filter_by_visibility("project")
print(f"Project-visible annotations: {len(project_annotations)}")

## Updating an annotation

The Annotation [`update`](https://py.d2s.org/annotation/#d2spy.models.annotation.Annotation.update) method can be used to modify an annotation's description, geometry, tags, visibility, or style.

In [ ]:
# Update the annotation description and tags
print(f"Before: {annotation.description}, tags={annotation.tags}")

annotation.update(
    description="Updated field boundary",
    tags=["boundary", "field-1", "verified"],
)

print(f"After: {annotation.description}, tags={annotation.tags}")

In [ ]:
# Change visibility to share with project members
annotation.update(visibility="project")
print(f"Visibility: {annotation.visibility}")

## Working with attachments

Annotations support file attachments such as images and videos. Supported file types: JPG, JPEG, PNG, GIF, WebP, MP4, MOV, WebM, and AVI.

The Annotation [`add_attachment`](https://py.d2s.org/annotation/#d2spy.models.annotation.Annotation.add_attachment) method uploads a file as an attachment to the annotation.

In [ ]:
# Upload a photo as an attachment
annotation.add_attachment("/full/path/to/field_photo.jpg")

After uploading, you can view the annotation's attachments by refreshing it with `get_annotation`. Each attachment includes metadata such as its `id`, `original_filename`, `content_type`, and `size_bytes`.

In [ ]:
# Refresh the annotation to see the updated attachments list
annotation = data_product.get_annotation(str(annotation.id))

# List all attachments
for attachment in annotation.attachments:
    print(f"ID: {attachment['id']}, File: {attachment['original_filename']}")

The Annotation [`download_attachment`](https://py.d2s.org/annotation/#d2spy.models.annotation.Annotation.download_attachment) method downloads an attachment to a local file path.

In [ ]:
# Download the first attachment
attachment_id = annotation.attachments[0]["id"]
annotation.download_attachment(attachment_id, "/full/path/to/downloaded_photo.jpg")

The Annotation [`delete_attachment`](https://py.d2s.org/annotation/#d2spy.models.annotation.Annotation.delete_attachment) method removes an attachment from the annotation.

In [ ]:
# Delete the attachment
annotation.delete_attachment(attachment_id)
print("Attachment deleted")

## Deleting an annotation

The Annotation [`delete`](https://py.d2s.org/annotation/#d2spy.models.annotation.Annotation.delete) method permanently removes the annotation and all of its attachments.

In [ ]:
# Delete the shared annotation we created earlier
shared_annotation.delete()
print("Annotation deleted")

Once finished working with annotations, you can revoke your authorization session by logging out.

In [ ]:
# Removes access token from future requests
workspace.logout()